# PGA Tournament Model: Champion Classifier

## Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

## Get Data

In [ ]:
import kagglehub
from kagglehub import KaggleDatasetAdapter

file_path = "tournament_shots_data.csv"

df = kagglehub.load_dataset(
  KaggleDatasetAdapter.PANDAS,
  "samanthastaheli/tournamentshots/versions/9",
  file_path,
  pandas_kwargs={"encoding": "latin1"} # European players have different spelling
)

df.head()

/tmp/ipykernel_3875/944555103.py:6: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  df = kagglehub.load_dataset(


100%|██████████| 18.4M/18.4M [00:00<00:00, 39.5MB/s]
/usr/local/lib/python3.12/dist-packages/kagglehub/pandas_datasets.py:92: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  result = read_function(


,tournament,tournament_id,player,player_id,round,hole,shot_number,shot_dist,to_hole,location,par,hole_yardage
0,THE PLAYERS Championship,R2023011,Ryan Armour,19803,1,1,1,246 yds,166 yds,Right Rough,4,423
1,THE PLAYERS Championship,R2023011,Ryan Armour,19803,1,1,2,163 yds,18 ft 5 in.,Green,4,423
2,THE PLAYERS Championship,R2023011,Ryan Armour,19803,1,1,3,20 ft 8 in.,2 ft 1 in.,Green,4,423
3,THE PLAYERS Championship,R2023011,Ryan Armour,19803,1,1,4,2 ft 1 in.,0,In Hole,4,423
4,THE PLAYERS Championship,R2023011,Ryan Armour,19803,1,2,1,234 yds,303 yds,Tree Outline,5,532


### Check Data

In [ ]:
# Group by player and tournament, then count unique rounds and holes
data_check = df.groupby(['tournament_id', 'player']).agg(
    rounds_played=('round', 'nunique'),
    unique_holes_played=('hole', 'nunique')
).reset_index()

# A complete tournament means exactly 4 rounds and 18 unique holes
# Flag anyone who doesn't match the 72-hole benchmark
is_complete = (data_check['rounds_played'] == 4) & (data_check['unique_holes_played'] == 18)
missing_data_players = data_check[~is_complete]

# Print the results
if missing_data_players.empty:
    print("All players have complete data (4 rounds, 18 holes each).")
else:
    print(f"Found {len(missing_data_players)} players with missing or incomplete data layers:")
    print(missing_data_players.to_string(index=False))

All players have complete data (4 rounds, 18 holes each).


### Change shotDist and toHole to float datapoints

In [ ]:
import re

def parse_golf_distance_to_yards(val):
    """
    Takes the 'shot_dist' pr 'to_hole' value and converts it to a numerical
    yardage (e.g. a value could be 334 yards converted to 334.0 or 5 ft 11 in.
    converted to 1.972).
    """
    # Convert to string and clean up whitespaces
    val = str(val).strip().lower()

    # Handle clean zeros or empty rows
    if val in ['0', '0.0', 'nan', '']:
        return 0.0

    # Check 1: If it's explicitly in yards (e.g., "334 yds")
    if 'yd' in val:
        match = re.search(r'([\d.]+)', val)
        return float(match.group(1)) if match else 0.0

    # Check 2: If it's in feet/inches (e.g., "27 ft 6 in.")
    if 'ft' in val or 'in' in val:
        # Extract feet if present
        ft_match = re.search(r'(\d+)\s*ft', val)
        feet = float(ft_match.group(1)) if ft_match else 0.0

        # Extract inches if present
        in_match = re.search(r'(\d+)\s*in', val)
        inches = float(in_match.group(1)) if in_match else 0.0

        # Convert total feet and inches into decimal yards (3 feet in a yard, 36 inches in a yard)
        total_yards = (feet / 3.0) + (inches / 36.0)
        return round(total_yards, 3) # Rounding to 3 decimal places for precision

    # Check 3: Fallback if it's a raw number string without units
    try:
        return float(val)
    except ValueError:
        return 0.0

# Apply the parsing function to both columns
df["shot_dist_yards"] = df["shot_dist"].apply(parse_golf_distance_to_yards)
df["to_hole_yards"] = df["to_hole"].apply(parse_golf_distance_to_yards)

print(df[["shot_dist", "shot_dist_yards", "to_hole", "to_hole_yards"]].head())

     shot_dist  shot_dist_yards      to_hole  to_hole_yards
0      246 yds          246.000      166 yds        166.000
1      163 yds          163.000  18 ft 5 in.          6.139
2  20 ft 8 in.            6.889   2 ft 1 in.          0.694
3   2 ft 1 in.            0.694            0          0.000
4      234 yds          234.000      303 yds        303.000


### Categorize Locations

In [ ]:
# Sort chronologically so shifts align perfectly within each hole
df = df.sort_values(by=['tournament_id', 'round', 'hole', 'player_id', 'shot_number']).copy()

# Get the previous landing location (where the current shot is being hit from)
df['shot_started_from'] = df.groupby(['tournament_id', 'round', 'hole', 'player_id'])['location'].shift(1)

# For the very first shot of a hole, the previous location is blank (NaN),
# which means they are hitting from the Tee Box.
df['shot_started_from'] = df['shot_started_from'].fillna('tee')

# Track the holed status as an independent variable
df["is_holed"] = df["location"].str.lower().str.contains("in hole", na=False)

In [ ]:
def categorize_locations(row):
    """
    Categorize a shot based off location type. Takes in a row/shot and returns
    the location type. The shot location types are tee box, approach, or putt.
    """
    start_loc = str(row["shot_started_from"]).lower().strip()
    end_loc = str(row["location"]).lower().strip()

    try:
        shot_num = int(float(row["shot_number"]))
    except (ValueError, TypeError):
        shot_num = None

    if shot_num == 1:
        shot_class = "Tee"
    elif "green" in start_loc:
        shot_class = "Putt"
    else:
        shot_class = "Approach"

    return shot_class

df["shot_type"] = df.apply(categorize_locations, axis=1)

df.head(10)

,tournament,tournament_id,player,player_id,round,hole,shot_number,shot_dist,to_hole,location,par,hole_yardage,shot_dist_yards,to_hole_yards,shot_started_from,is_holed,shot_type
40935,THE PLAYERS Championship,R2023011,Jerry Kelly,8075,1,1,1,235 yds,176 yds,Right Fairway Bunker,4,423,235.000,176.000,tee,False,Tee
40936,THE PLAYERS Championship,R2023011,Jerry Kelly,8075,1,1,2,170 yds,30 ft 5 in.,Green,4,423,170.000,10.139,Right Fairway Bunker,False,Approach
40937,THE PLAYERS Championship,R2023011,Jerry Kelly,8075,1,1,3,34 ft 8 in.,3 ft 11 in.,Green,4,423,11.556,1.306,Green,False,Putt
40938,THE PLAYERS Championship,R2023011,Jerry Kelly,8075,1,1,4,3 ft 11 in.,0,In Hole,4,423,1.306,0.000,Green,True,Putt
0,THE PLAYERS Championship,R2023011,Ryan Armour,19803,1,1,1,246 yds,166 yds,Right Rough,4,423,246.000,166.000,tee,False,Tee
1,THE PLAYERS Championship,R2023011,Ryan Armour,19803,1,1,2,163 yds,18 ft 5 in.,Green,4,423,163.000,6.139,Right Rough,False,Approach
2,THE PLAYERS Championship,R2023011,Ryan Armour,19803,1,1,3,20 ft 8 in.,2 ft 1 in.,Green,4,423,6.889,0.694,Green,False,Putt
3,THE PLAYERS Championship,R2023011,Ryan Armour,19803,1,1,4,2 ft 1 in.,0,In Hole,4,423,0.694,0.000,Green,True,Putt
309,THE PLAYERS Championship,R2023011,Stewart Cink,20229,1,1,1,280 yds,132 yds,Right Rough,4,423,280.000,132.000,tee,False,Tee
310,THE PLAYERS Championship,R2023011,Stewart Cink,20229,1,1,2,114 yds,62 ft 10 in.,Right Fairway,4,423,114.000,20.944,Right Rough,False,Approach


In [ ]:
def categorize_positions(row):
    """
    Categorize shots based on where the shot starts from. Takes in row/shot and
    returns the shots start position. The positions are tee, putt, fairway,
    rough, sand, and approach.
    """
    start_loc = str(row["shot_started_from"]).lower().strip()
    end_loc = str(row["location"]).lower().strip()

    if row["shot_number"] == 1:
        shot_pos = "tee"
    elif "green" in start_loc:
        shot_pos = "putt"
    elif "fairway" in start_loc:
        shot_pos = "fairway"
    elif "rough" in start_loc:
        shot_pos = "rough"
    elif "bunker" in start_loc:
        shot_pos = "sand"
    else:
        shot_pos = "approach"

    return shot_pos

df["shot_pos"] = df.apply(categorize_positions, axis=1)

df.head(10)

,tournament,tournament_id,player,player_id,round,hole,shot_number,shot_dist,to_hole,location,par,hole_yardage,shot_dist_yards,to_hole_yards,shot_started_from,is_holed,shot_type,shot_pos
40935,THE PLAYERS Championship,R2023011,Jerry Kelly,8075,1,1,1,235 yds,176 yds,Right Fairway Bunker,4,423,235.000,176.000,tee,False,Tee,approach
40936,THE PLAYERS Championship,R2023011,Jerry Kelly,8075,1,1,2,170 yds,30 ft 5 in.,Green,4,423,170.000,10.139,Right Fairway Bunker,False,Approach,fairway
40937,THE PLAYERS Championship,R2023011,Jerry Kelly,8075,1,1,3,34 ft 8 in.,3 ft 11 in.,Green,4,423,11.556,1.306,Green,False,Putt,putt
40938,THE PLAYERS Championship,R2023011,Jerry Kelly,8075,1,1,4,3 ft 11 in.,0,In Hole,4,423,1.306,0.000,Green,True,Putt,putt
0,THE PLAYERS Championship,R2023011,Ryan Armour,19803,1,1,1,246 yds,166 yds,Right Rough,4,423,246.000,166.000,tee,False,Tee,approach
1,THE PLAYERS Championship,R2023011,Ryan Armour,19803,1,1,2,163 yds,18 ft 5 in.,Green,4,423,163.000,6.139,Right Rough,False,Approach,rough
2,THE PLAYERS Championship,R2023011,Ryan Armour,19803,1,1,3,20 ft 8 in.,2 ft 1 in.,Green,4,423,6.889,0.694,Green,False,Putt,putt
3,THE PLAYERS Championship,R2023011,Ryan Armour,19803,1,1,4,2 ft 1 in.,0,In Hole,4,423,0.694,0.000,Green,True,Putt,putt
309,THE PLAYERS Championship,R2023011,Stewart Cink,20229,1,1,1,280 yds,132 yds,Right Rough,4,423,280.000,132.000,tee,False,Tee,approach
310,THE PLAYERS Championship,R2023011,Stewart Cink,20229,1,1,2,114 yds,62 ft 10 in.,Right Fairway,4,423,114.000,20.944,Right Rough,False,Approach,rough


### Drop Shot Number Nan

In [ ]:
# Force shot_number to be numeric, turning rogue text/strings into NaN safely
df["shot_number"] = pd.to_numeric(df["shot_number"], errors="coerce")

# Drop rows where shot_number became NaN to protect the max calculation
df = df.dropna(subset=["shot_number"])

df.head()

,tournament,tournament_id,player,player_id,round,hole,shot_number,shot_dist,to_hole,location,par,hole_yardage,shot_dist_yards,to_hole_yards,shot_started_from,is_holed,shot_type,shot_pos
40935,THE PLAYERS Championship,R2023011,Jerry Kelly,8075,1,1,1.0,235 yds,176 yds,Right Fairway Bunker,4,423,235.000,176.000,tee,False,Tee,approach
40936,THE PLAYERS Championship,R2023011,Jerry Kelly,8075,1,1,2.0,170 yds,30 ft 5 in.,Green,4,423,170.000,10.139,Right Fairway Bunker,False,Approach,fairway
40937,THE PLAYERS Championship,R2023011,Jerry Kelly,8075,1,1,3.0,34 ft 8 in.,3 ft 11 in.,Green,4,423,11.556,1.306,Green,False,Putt,putt
40938,THE PLAYERS Championship,R2023011,Jerry Kelly,8075,1,1,4.0,3 ft 11 in.,0,In Hole,4,423,1.306,0.000,Green,True,Putt,putt
0,THE PLAYERS Championship,R2023011,Ryan Armour,19803,1,1,1.0,246 yds,166 yds,Right Rough,4,423,246.000,166.000,tee,False,Tee,approach


## Helper Functions

In [ ]:
def get_hole_scores(df):
    """
    Calculates hole scores using .max() on the hole. Returns a column of hole
    scores for every shot called 'total_shots'.
    """
    # Group by player, round, and hole, then find the highest shot number
    shots_per_hole = df.groupby(['tournament_id', 'player', 'round', 'hole'])['shot_number'].max().reset_index()

    # Rename the column to make it clear it represents the hole score
    shots_per_hole.rename(columns={'shot_number': 'total_shots'}, inplace=True)
    return shots_per_hole

def get_total_scores(df):
    """
    Calculate total tournament score using `get_hole_scores` and .sum() on
    the hole scores. Returns a column of tournamet total score called
    'total_strokes'.
    """
    hole_scores = get_hole_scores(df)

    # Sum the hole scores to get the total tournament score for each player
    total_scores = hole_scores.groupby(['player', 'tournament_id'])['total_shots'].sum().reset_index()
    total_scores.rename(columns={'total_shots': 'total_strokes'}, inplace=True)

    return total_scores

## Stats between tournaments and players

In [ ]:
def get_drive_accuracy_by_tour_and_player(df):
    """
    Calculates players drive accuracy grouped by tournment and player. Returns
    a dataframe with the new column 'driving_accuracy_%'.
    """
    # Get drives only
    drives_df = df[(df['shot_type'] == 'Tee') & (df['par'] != 3)].copy()

    # If fairway in location then True, else False
    drives_df['hit_fairway'] = drives_df['location'].str.lower().str.contains('fairway', na=False)

    # Get new df with driving accuracy and sort by driving accuracy
    player_accuracy_df = pd.DataFrame({
        'driving_accuracy_%': (drives_df.groupby(['tournament_id', 'player'])['hit_fairway'].mean() * 100).round(2),
        'total_drives_tracked': drives_df.groupby(['tournament_id', 'player'])['hit_fairway'].count(),
        'avg_drive_distance': drives_df.groupby(['tournament_id', 'player'])['shot_dist_yards'].mean()
    }).reset_index()
    player_accuracy_df = player_accuracy_df.sort_values(by='driving_accuracy_%', ascending=False).reset_index(drop=True)

    return player_accuracy_df

def get_gir_by_tour_and_player(df):
    """
    Calculates if a player gets the  green in regulation (gir) grouped by
    tournment and player. Returns a dataframe with the new column 'GIR'.
    """
    green_shots = df[df['location'].str.lower().str.contains('green', na=False)].copy()

    # Find the first shot that hit the green for every player on every hole
    # (Using groupby + min ensures we catch the exact shot number they reached the surface)
    first_green_shot = green_shots.groupby(['tournament_id', 'round', 'hole', 'player', 'par'])['shot_number'].min().reset_index()
    first_green_shot.rename(columns={'shot_number': 'shot_reached_green'}, inplace=True)

    # Apply the official GIR condition: shot_reached_green <= (par - 2)
    first_green_shot['GIR'] = first_green_shot['shot_reached_green'] <= (first_green_shot['par'] - 2)

    return first_green_shot

def get_gir_percentage_by_tour_and_player(df):
    """
    Calculates the percentage a player gets the  green in regulation (gir)
    grouped by tournment and player. Returns a dataframe with the new
    column 'GIR_%'.
    """
    gir = get_gir_by_tour_and_player(df)

    # Calculate final GIR %
    player_gir_df = gir.groupby(['tournament_id', 'player'])['GIR'].mean().reset_index()

    # Get the percentage
    player_gir_df['GIR_%'] = (player_gir_df['GIR'] * 100).round(2)
    player_gir_df = player_gir_df.sort_values(by='GIR_%', ascending=False).reset_index(drop=True)

    # Clean up temporary aggregation column
    player_gir_df.drop(columns=['GIR'], inplace=True)
    return player_gir_df

def get_avg_proximity_to_hole_after_approach_shot_by_tour_and_player(df):
    """
    Calculates the average remaining distance to the hole for approach shots.
    Returns a dataframe with the new column
    'avg_proximity_after_approach_shot_feet'.
    """
    # Get next shot remaining distance
    # Make sure sorted
    df_sorted = df.sort_values(by=['tournament_id', 'round', 'hole', 'player', 'shot_number']).copy()

    # Get the starting distance of the next shot (the remaining distance after current shot)
    df_sorted['remaining_dist_after_shot'] = df_sorted.groupby(['tournament_id', 'round', 'hole', 'player'])['to_hole_yards'].shift(-1)

    # Filter Approach Shots
    approach_shots = df_sorted[df_sorted['shot_type'].str.lower().str.contains('approach', na=False)].copy()

    # Create new df
    player_proximity_df = pd.DataFrame({
        'avg_proximity_after_approach_shot_yards': approach_shots.groupby(['tournament_id', 'player'])['remaining_dist_after_shot'].mean(),
        'approach_shots_measured': approach_shots.groupby(['tournament_id', 'player'])['remaining_dist_after_shot'].count()
    }).reset_index()

    # Sort from closest proximity (best) to furthest away (worst)
    player_proximity_df = player_proximity_df.sort_values(by='avg_proximity_after_approach_shot_yards', ascending=True).reset_index(drop=True)

    # Clean up any players with low sample sizes if necessary
    leaderboard_proximity = player_proximity_df[player_proximity_df['approach_shots_measured'] >= 5]

    # convert to feet
    leaderboard_proximity['avg_proximity_after_approach_shot_feet'] = (leaderboard_proximity['avg_proximity_after_approach_shot_yards'] * 3).round(1)
    return leaderboard_proximity

def get_scrambling_percentage_by_tour_and_player(df):
    """
    Calculates the percentage a player makes par after missing the green in
    regulation (GIR). Reeturns a datafrane with a new column 'scrambling_%'.
    """
    # Identify missed GIR and final scores
    hole_scores = get_hole_scores(df)
    green_shots = df[df['location'].str.lower().str.contains('green', na=False)].copy()
    first_green_shot = green_shots.groupby(['tournament_id', 'round', 'hole', 'player', 'par'])['shot_number'].min().reset_index()
    first_green_shot.rename(columns={'shot_number': 'shot_reached_green'}, inplace=True)

    # Merge hole scores and green data together
    scramble_base = pd.merge(hole_scores, first_green_shot, on=['tournament_id', 'round', 'hole', 'player'], how='left')

    # If 'shot_reached_green' is NaN, it means they never hit the green at all
    # fill those with a high dummy number so it safely counts as a missed GIR.
    scramble_base['shot_reached_green'] = scramble_base['shot_reached_green'].fillna(99)

    # Run scrambling flags

    # Flag 1: Did the player miss the green in regulation?
    scramble_base['missed_GIR'] = scramble_base['shot_reached_green'] > (scramble_base['par'] - 2)

    # Flag 2: Did the player make par or better?
    scramble_base['saved_par'] = scramble_base['total_shots'] <= scramble_base['par']

    # Isolate ONLY the opportunities where the player actually missed the green
    scramble_opportunities = scramble_base[scramble_base['missed_GIR'] == True].copy()

    # Compile scrambling dataframe

    # Create the new standalone DataFrame
    player_scrambling_df = pd.DataFrame({
        'scrambling_%': (scramble_opportunities.groupby(['tournament_id', 'player'])['saved_par'].mean() * 100).round(2),
        'scramble_opportunities': scramble_opportunities.groupby(['tournament_id', 'player'])['saved_par'].count(),
        'scramble_saves': scramble_opportunities.groupby(['tournament_id', 'player'])['saved_par'].sum()
    }).reset_index()

    # Sort from best scrambler to worst
    player_scrambling_df = player_scrambling_df.sort_values(by='scrambling_%', ascending=False).reset_index(drop=True)
    return player_scrambling_df

def get_player_sand_saves_by_tour_and_player(df):
    """
    Calculates the percentage a player makes par or less after they have land
    in the sand. Returns a dataframe with a new column 'sand_save_%'.
    """
    hole_scores = get_hole_scores(df)

    # Make sure shots are in order
    df_sorted = df.sort_values(by=['tournament_id', 'round', 'hole', 'player', 'shot_number']).copy()

    # Look at the next shot's location using .shift(-1)
    df_sorted['next_location'] = df_sorted.groupby(['tournament_id', 'round', 'hole', 'player'])['location'].shift(-1)

    # Filter for shots hit from a bunker that landed on the green (Using 'bunker' catches green-side sand shots)
    sand_to_green = df_sorted[
        df_sorted['location'].str.lower().str.contains('bunker', na=False) &
        df_sorted['next_location'].str.lower().str.contains('green', na=False)
    ].copy()

    # Mark these holes as having a valid sand-save opportunity
    sand_to_green['had_sand_opportunity'] = True

    # Keep just one record per hole for players who hit out of the sand to the green
    sand_opportunities = sand_to_green.drop_duplicates(subset=['round', 'hole', 'player', 'par'])

    # Merge and flag sand saves
    sand_base = pd.merge(
        hole_scores,
        sand_opportunities[['tournament_id', 'round', 'hole', 'player', 'had_sand_opportunity', 'par']],
        on=['tournament_id', 'round', 'hole', 'player'],
        how='inner' # drops any holes where they never went bunker-to-green
    )

    # A sand save is successful if their final score is less than or equal to par
    sand_base['is_sand_save'] = sand_base['total_shots'] <= sand_base['par']

    # Create leaderboard
    player_sand_saves_df = pd.DataFrame({
        'sand_save_%': (sand_base.groupby(['tournament_id', 'player'])['is_sand_save'].mean() * 100).round(2),
        'sand_opportunities': sand_base.groupby(['tournament_id', 'player'])['is_sand_save'].count(),
        'sand_saves_made': sand_base.groupby(['tournament_id', 'player'])['is_sand_save'].sum()
    }).reset_index()

    # Sort the table from best sand-scrambler to worst
    return player_sand_saves_df.sort_values(by='sand_save_%', ascending=False).reset_index(drop=True)

def get_player_putts_per_gir_by_tour_and_player(df):
    """
    The average amount of putts a player takes when the make the green in
    regulation (GIR). Returns a dataframe with a new column 'putts_per_gir'.
    """
    # Get putts
    putts_df = df[df['shot_type'] == 'Putt'].copy()

    # Group by player and round to count their total putts
    round_putts = putts_df.groupby(["tournament_id", "round", "player"]).size().reset_index(name="total_putts")

    # Get average putts per round
    player_putts_per_round = round_putts.groupby(["tournament_id", "player"])["total_putts"].mean().reset_index(name="putts_per_round")
    player_putts_per_round["putts_per_round"] = player_putts_per_round["putts_per_round"].round(2)

    # Get Putts per GIR

    # Count the number of putts taken on every hole
    hole_putts = putts_df.groupby(["tournament_id", "round", "hole", "player"]).size().reset_index(name='hole_putt_count')

    # Get 'GIR' boolean column (True/False)
    gir_df = get_gir_by_tour_and_player(df)
    gir_putts_base = pd.merge(
        gir_df[['tournament_id', 'round', 'hole', 'player', 'GIR']],
        hole_putts,
        on=['tournament_id', 'round', 'hole', 'player'],
        how='left'
    )

    # Fill NaN with 0 for holes where they holed out from off the green and took 0 putts
    gir_putts_base['hole_putt_count'] = gir_putts_base['hole_putt_count'].fillna(0)

    # Isolate only the holes where the player successfully made a GIR
    gir_only_holes = gir_putts_base[gir_putts_base['GIR'] == True]

    # Calculate the final average putts per GIR
    # player_putts_per_gir = gir_only_holes.groupby(['tournament_id', 'player']).size().reset_index(name='putts_per_gir') # total counts
    return gir_only_holes.groupby(['tournament_id', 'player'])['hole_putt_count'].mean().reset_index(name='putts_per_gir') # average

def get_putt_percentages_by_tour_and_player(df):
    """
    Calculates the perctantage of 1, 2, 3, and 3+ putts. Returns a dataframe
    with 4 new columns for each putt scenario caclulates. The columns are named
    '1_putt_%', '2_putt_%', '3_putt_%', and '3+_putt_%'.
    """
    # Get putts
    putts_df = df[df['shot_type'] == 'Putt'].copy()

    hole_putts = putts_df.groupby(["tournament_id", "round", "hole", "par", "player"]).size().reset_index(name="total_putts")
    hole_putts.head()

    # Create boolean flags for putt numbers
    hole_putts['is_1_putt'] = hole_putts['total_putts'] == 1
    hole_putts['is_2_putt'] = hole_putts['total_putts'] == 2
    hole_putts['is_3_putt'] = hole_putts['total_putts'] == 3
    hole_putts['is_3+_putt'] = hole_putts['total_putts'] > 3

    # Calculate percentage of 1, 2, 3, 3+ putts
    return pd.DataFrame({
        '1_putt_%': (hole_putts.groupby(['tournament_id', 'player'])['is_1_putt'].mean() * 100).round(2),
        '2_putt_%': (hole_putts.groupby(['tournament_id', 'player'])['is_2_putt'].mean() * 100).round(2),
        '3_putt_%': (hole_putts.groupby(['tournament_id', 'player'])['is_3_putt'].mean() * 100).round(2),
        '3+_putt_%': (hole_putts.groupby(['tournament_id', 'player'])['is_3+_putt'].mean() * 100).round(2),
    }).reset_index()

def get_putt_make_shot_dist_percentages_by_tour_and_player(df):
    """
    Calculates the perctange of putts made in the hole from under 5 feet, 5-10
    feet, and over 10 feet. Returns a dataframe with 3 new columns for every
    putt distance category. The column names are 'under_5_make_%',
    '5_to_10_make_%', and 'over_10_make_%'.
    """
    putts_only_df = df[df["shot_type"] == "Putt"].copy()
    putts_only_df["is_made"] = putts_only_df["location"].str.lower().str.contains("in hole", na=False)

    # Create boolean flags for putt distances
    putts_only_df['is_under_5'] = putts_only_df['to_hole_yards'] > 5/3
    putts_only_df['is_5_10'] = (putts_only_df['to_hole_yards'] <= 5/3) & (putts_only_df['to_hole_yards'] >= 10/3)
    putts_only_df['is_over_10'] = putts_only_df['to_hole_yards'] < 10/3

    # Calculate percentage of 1, 2, 3, 3+ putts
    putt_make_percentages = pd.DataFrame({
        'under_5_make_%': (putts_only_df.groupby(['tournament_id', 'player'])['is_under_5'].mean() * 100).round(2),
        '5_to_10_make_%': (putts_only_df.groupby(['tournament_id', 'player'])['is_5_10'].mean() * 100).round(2),
        'over_10_make_%': (putts_only_df.groupby(['tournament_id', 'player'])['is_over_10'].mean() * 100).round(2),
    }).reset_index()

    # Clean up any players who had 0 putts in a specific distance bin (fills NaN with 0)
    percentage_cols = ['under_5_make_%', '5_to_10_make_%', 'over_10_make_%']
    putt_make_percentages[percentage_cols] = putt_make_percentages[percentage_cols].fillna(0.0)

    # Sort by the best short-range putters
    return putt_make_percentages.sort_values(by='under_5_make_%', ascending=False).reset_index(drop=True)

def get_avg_in_hole_shot_distance_by_tour_and_player(df):
    """
    Calculates the average distance of the in hole shots grouped by tournament
    and player. Returns a dataframe with a new column
    'avg_in_hole_shot_dist_feet'.
    """
    hole_shots = df[df['location'] == 'In Hole'].copy()

    avg_in_hole_dist = hole_shots.groupby(["tournament_id", "player"])["shot_dist_yards"].mean().reset_index(name="avg_in_hole_shot_dist")

    # convert yards to feet
    avg_in_hole_dist['avg_in_hole_shot_dist_feet'] = (avg_in_hole_dist['avg_in_hole_shot_dist'] * 3).round(2)

    # sort
    return avg_in_hole_dist.sort_values(by="avg_in_hole_shot_dist_feet", ascending=False)

## Prepare Data

In [ ]:
def get_player_stats_df(df):
    """
    Creates a dataframe of player stats using the helper functions defined above
    in stats between tournanamnets and players. Returns a merged dataframe of
    every player stats calculated in the helper functions.
    """
    # New players stats df
    player_df = (
        df[["tournament_id", "player", "player_id"]]
        .drop_duplicates()
        .sort_values(by=["tournament_id", "player"])
        .reset_index(drop=True)
    )

    # Get drives, approach, short game, putting, and in hole dist
    drive_accuracy = get_drive_accuracy_by_tour_and_player(df)
    gir = get_gir_percentage_by_tour_and_player(df)
    proximity_to_hole = get_avg_proximity_to_hole_after_approach_shot_by_tour_and_player(df)
    scrambling = get_scrambling_percentage_by_tour_and_player(df)
    sand_saves = get_player_sand_saves_by_tour_and_player(df)
    putts_per_gir = get_player_putts_per_gir_by_tour_and_player(df)
    putt_percentages = get_putt_percentages_by_tour_and_player(df)
    putt_make_percentages = get_putt_make_shot_dist_percentages_by_tour_and_player(df)
    in_hole_dist = get_avg_in_hole_shot_distance_by_tour_and_player(df)

    # Get total strokes
    total_strokes = get_total_scores(df)

    # Merge dataframes
    player_stats = (
        player_df
        .merge(drive_accuracy, on=["player", "tournament_id"], how="left")
        .merge(gir, on=["player", "tournament_id"], how="left")
        .merge(proximity_to_hole, on=["player", "tournament_id"], how="left")
        .merge(scrambling, on=["player", "tournament_id"], how="left")
        .merge(sand_saves, on=["player", "tournament_id"], how="left")
        .merge(putts_per_gir, on=["player", "tournament_id"], how="left")
        .merge(putt_percentages, on=["player", "tournament_id"], how="left")
        .merge(putt_make_percentages, on=["player", "tournament_id"], how="left")
        .merge(in_hole_dist, on=["player", "tournament_id"], how="left")
        .merge(total_strokes, on=["player", "tournament_id"], how="left")
    )
    return player_stats


# get the players championship data
# Using the players championship because have multiple years of data

# Get players championship 2021-2025
train_tour_ids = [f"R{year}011" for year in range(2021, 2026)]
train_years_only_df = df[df["tournament_id"].isin(train_tour_ids)]
# train_years_only_df = df[df["tournament_id"] == "R2024011"] # TODO switch back once have all data
test_years_only_df = df[df["tournament_id"] == "R2026011"]
# the_players_champ_df = df[df["tournament"] == "THE PLAYERS Championship"]

player_stats = get_player_stats_df(train_years_only_df)
player_stats_test = get_player_stats_df(test_years_only_df)
player_stats.head()

,tournament_id,player,player_id,driving_accuracy_%,total_drives_tracked,avg_drive_distance,GIR_%,avg_proximity_after_approach_shot_yards,approach_shots_measured,avg_proximity_after_approach_shot_feet,...,1_putt_%,2_putt_%,3_putt_%,3+_putt_%,under_5_make_%,5_to_10_make_%,over_10_make_%,avg_in_hole_shot_dist,avg_in_hole_shot_dist_feet,total_strokes
0,R2023011,Aaron Baddeley,22371,67.86,56,274.142857,61.97,2.891039,103,8.7,...,39.44,53.52,7.04,0.00,5.04,0.0,97.48,1.194458,3.58,295.0
1,R2023011,Aaron Rai,46414,78.57,56,280.910714,74.65,2.154110,91,6.5,...,42.25,50.70,4.23,2.82,8.40,0.0,97.48,3.087556,9.26,282.0
2,R2023011,Aaron Wise,49964,76.79,56,281.660714,40.28,8.417056,126,25.3,...,52.11,42.25,5.63,0.00,7.34,0.0,92.66,1.605264,4.82,308.0
3,R2023011,Adam Hadwin,33399,73.21,56,280.892857,72.22,1.651205,88,5.0,...,41.67,48.61,9.72,0.00,8.26,0.0,97.52,1.299056,3.90,281.0
4,R2023011,Adam Long,35449,66.07,56,274.964286,54.17,3.455098,112,10.4,...,38.89,48.61,12.50,0.00,9.60,0.0,96.00,1.605681,4.82,309.0


In [ ]:
# Get winners column
winners_dict = {
    "R2021011": "Justin Thomas",
    "R2022011": "Cameron Smith",
    "R2023011": "Scottie Scheffler",
    "R2024011": "Scottie Scheffler",
    "R2025011": "Rory McIlroy",
    "R2026011": "Cameron Young"
}

clean_winners = {tid: name.lower().strip() for tid, name in winners_dict.items()}

# Map and apply to train set
mapped_train = player_stats['tournament_id'].map(clean_winners)
player_stats['is_winner'] = (player_stats['player'].str.lower().str.strip() == mapped_train).astype(int)

# Map and apply to test set
mapped_test = player_stats_test['tournament_id'].map(clean_winners)
player_stats_test['is_winner'] = (player_stats_test['player'].str.lower().str.strip() == mapped_test).astype(int)

In [ ]:
# Confrim winners are there
print(player_stats.groupby('tournament_id')['is_winner'].any().astype(int))
print(player_stats_test.groupby('tournament_id')['is_winner'].any().astype(int))

tournament_id
R2023011    1
R2024011    1
R2025011    1
Name: is_winner, dtype: int64
tournament_id
R2026011    1
Name: is_winner, dtype: int64


## Neural Network: Champion Classifier

Predict who will win the 2026 Player Championship given the data provided.

In [ ]:
player_stats.columns

Index(['tournament_id', 'player', 'player_id', 'driving_accuracy_%',
       'total_drives_tracked', 'avg_drive_distance', 'GIR_%',
       'avg_proximity_after_approach_shot_yards', 'approach_shots_measured',
       'avg_proximity_after_approach_shot_feet', 'scrambling_%',
       'scramble_opportunities', 'scramble_saves', 'sand_save_%',
       'sand_opportunities', 'sand_saves_made', 'putts_per_gir', '1_putt_%',
       '2_putt_%', '3_putt_%', '3+_putt_%', 'under_5_make_%', '5_to_10_make_%',
       'over_10_make_%', 'avg_in_hole_shot_dist', 'avg_in_hole_shot_dist_feet',
       'total_strokes', 'is_winner'],
      dtype='object')

In [ ]:
# dataset for NN

# Define features
# bad features: avg_driving_distance
# feature_cols = [ # all features included
#     'driving_accuracy_%',
#     'total_drives_tracked', 'avg_drive_distance', 'GIR_%',
#     'avg_proximity_after_approach_shot_yards', 'approach_shots_measured',
#     'avg_proximity_after_approach_shot_feet', 'scrambling_%',
#     'scramble_opportunities', 'scramble_saves', 'sand_save_%',
#     'sand_opportunities', 'sand_saves_made', 'putts_per_gir', '1_putt_%',
#     '2_putt_%', '3_putt_%', '3+_putt_%', 'under_5_make_%', '5_to_10_make_%',
#     'over_10_make_%', 'avg_in_hole_shot_dist', 'avg_in_hole_shot_dist_feet',
#     'total_strokes'
# ]
# Best combination of features
feature_cols = [
    'driving_accuracy_%', 'GIR_%', 'putts_per_gir', 'total_strokes',
    'under_5_make_%', '5_to_10_make_%', 'over_10_make_%',
]


X_train = player_stats[feature_cols].values
y_train = player_stats['is_winner'].values.astype(np.float32)
X_test = player_stats_test[feature_cols].values
y_test = player_stats_test['is_winner'].values.astype(np.float32)

# Neural networks are highly sensitive to unscaled data (e.g., strokes vs percentages)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_train)

# Convert your training and testing splits into PyTorch tensors
X_train_t = torch.FloatTensor(X_train)
y_train_t = torch.FloatTensor(y_train).unsqueeze(1)
X_test_t = torch.FloatTensor(X_test)
y_test_t = torch.FloatTensor(y_test).unsqueeze(1)

# Handle class imbalance: Calculate the weight of non-winners vs winners
num_negatives = (y_train == 0).sum()
num_positives = (y_train == 1).sum()
pos_weight = torch.tensor([num_negatives / max(num_positives, 1)])

In [ ]:
class GolfWinnerClassifier(nn.Module):
    def __init__(self, input_dim):
        super(GolfWinnerClassifier, self).__init__()

        # Layer 1: Input -> Hidden (16 neurons)
        self.fc1 = nn.Linear(input_dim, 16)
        self.bn1 = nn.BatchNorm1d(16)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.2)

        # Layer 2: Hidden -> Hidden (8 neurons)
        self.fc2 = nn.Linear(16, 8)
        self.bn2 = nn.BatchNorm1d(8)

        # Layer 3: Hidden -> Output (1 raw value / logit)
        self.fc3 = nn.Linear(8, 1)

    def forward(self, x):
        x = self.fc1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.dropout(x)
        x = self.fc2(x)
        x = self.bn2(x)
        x = self.relu(x)
        x = self.dropout(x)
        x = self.fc3(x)
        return x

In [ ]:
epochs = 120
batch_size = 16
model = GolfWinnerClassifier(input_dim=X_train.shape[1])
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = optim.Adam(model.parameters(), lr=0.003, weight_decay=1e-4)

In [ ]:
print("--- STARTING TRAINING LOOP ---")
model.train()

for epoch in range(epochs):
    # Shuffle the dataset indices every epoch
    permutation = torch.randperm(X_train_t.size()[0])
    epoch_loss = 0

    for i in range(0, X_train_t.size()[0], batch_size):
        indices = permutation[i:i+batch_size]
        batch_x, batch_y = X_train_t[indices], y_train_t[indices]

        # 1. Clear out old gradients
        optimizer.zero_grad()

        # 2. Forward Pass (Get raw outputs / logits)
        outputs = model(batch_x)
        loss = criterion(outputs, batch_y)

        # 3. Backward Pass (Calculate gradients)
        loss.backward()

        # 4. Optimize (Update weights)
        optimizer.step()

        epoch_loss += loss.item()

    # Print progress every 20 epochs
    if (epoch + 1) % 20 == 0:
        avg_loss = epoch_loss / (X_train_t.size()[0] / batch_size)
        print(f"Epoch [{epoch+1}/{epochs}] -> Average Loss: {avg_loss:.4f}")

--- STARTING TRAINING LOOP ---
Epoch [20/120] -> Average Loss: 0.7676
Epoch [40/120] -> Average Loss: 0.3199
Epoch [60/120] -> Average Loss: 0.4192
Epoch [80/120] -> Average Loss: 0.5691
Epoch [100/120] -> Average Loss: 0.3278
Epoch [120/120] -> Average Loss: 0.2026


In [ ]:
print("\n--- RUNNING EVALUATION ON TEST DATA ---")
model.eval() # Set model to evaluation mode (turns off dropout/batchnorm updates)

with torch.no_grad():
    # 1. Get raw predictions for test features
    test_logits = model(X_test_t)

    # 2. Convert raw logits into percentages/probabilities using Sigmoid
    probabilities = torch.sigmoid(test_logits)

    # 3. Convert probabilities to hard binary decisions (1 if > 50%, else 0)
    predictions = (probabilities >= 0.5).float().numpy()

# 4. Convert the actual test targets back to a numpy array for comparison
y_true = y_test_t.numpy()

# 5. Print comprehensive performance metrics
print("\n--- CONFUSION MATRIX ---")
print(confusion_matrix(y_true, predictions))

print("\n--- CLASSIFICATION REPORT ---")
print(classification_report(y_true, predictions, target_names=["Non-Winner", "Winner"]))


--- RUNNING EVALUATION ON TEST DATA ---

--- CONFUSION MATRIX ---
[[117   4]
 [  0   1]]

--- CLASSIFICATION REPORT ---
              precision    recall  f1-score   support

  Non-Winner       1.00      0.97      0.98       121
      Winner       0.20      1.00      0.33         1

    accuracy                           0.97       122
   macro avg       0.60      0.98      0.66       122
weighted avg       0.99      0.97      0.98       122



In [ ]:
# --- Get winner results ---

# 1. Map probabilities array back to the original test tracker DataFrame
# (Using player_stats_test from your split step)
results_df = player_stats_test.copy()
results_df['win_probability'] = probabilities.numpy().flatten()

# 2. Sort the field to see who the model ranked at the top
ranked_field = results_df.sort_values(by='win_probability', ascending=False).reset_index(drop=True)

# 3. Print the top 5 predicted contenders
print("--- MODEL'S TOP TOURNAMENT PREDICTIONS ---")
print(ranked_field[['player', 'tournament_id', 'win_probability']].head(5))

--- MODEL'S TOP TOURNAMENT PREDICTIONS ---
              player tournament_id  win_probability
0      Cameron Young      R2026011         0.908309
1  Xander Schauffele      R2026011         0.771262
2      Ludvig Ãberg      R2026011         0.586483
3        Sepp Straka      R2026011         0.555737
4     Viktor Hovland      R2026011         0.509794


In [ ]:
# Save model
from google.colab import files

torch.save(model.state_dict(), 'model_champion_classifier.pt')

files.download('model_champion_classifier.pt')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>